# Verify the dashboard numbers by hand

Checks that what the site shows is what the models actually produce. Three independent levels:

| Level | Needs | Question answered |
|---|---|---|
| **A. Files served** | the two `*_performance.json` only | Is the deployed site serving the files in the repository? Are the numbers in those files internally coherent (counts, rates, AUC from the ROC, DCA, histograms)? |
| **B. Recomputation** | the individual-level predictions (`algoX_pred.csv`, private) | Do AUC, CI, ROC, threshold table, calibration, DCA, histograms, zones match a recomputation written here from scratch with scikit-learn/numpy (not `performance.py`)? |
| **C. Browser engine** | node | Does the JavaScript model (what visitors run) give the same probabilities and SHAP values as the Python model? |

Level B reads only files in your private folder; nothing individual-level goes into the repository.

## 0. Setup

In [ ]:
import json, hashlib, subprocess, shutil, urllib.request, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression

DASH    = Path("..").resolve()                                   # <-- the repository folder (this notebook lives in tools/)
PRIVATE = DASH.parent / "dashboard_private"                          # <-- folder with algo1_pred.csv / algo2_pred.csv
SITE    = "https://tdupont.github.io/ptsd-dashboard/"                # deployed site ("" to skip the online check)
ALGOS   = {"algo1": "Algorithm 1 (XGBoost, held-out test set)", "algo2": "Algorithm 2 (L2 logistic, 5-fold OOF)"}

PERF = {a: json.loads((DASH / "docs/models" / f"{a}_performance.json").read_text()) for a in ALGOS}
_trapz = getattr(np, "trapezoid", None) or np.trapz   # numpy < 2 has no np.trapezoid
results = []   # (level, algo, check, shown, recomputed, ok)
def record(level, algo, check, shown, mine, tol=0.0, exact=False):
    if exact: ok = (shown == mine)
    else:
        a, b = np.asarray(shown, float), np.asarray(mine, float)
        ok = bool(np.all(np.abs(a - b) <= tol)) if a.shape == b.shape else False
    results.append((level, algo, check, shown if np.ndim(shown) == 0 else f"[{np.size(shown)} values]",
                    mine if np.ndim(mine) == 0 else f"[{np.size(mine)} values]", ok))
    return ok
print({a: (p["n"], p["n_events"], p["created"]) for a, p in PERF.items()})

## A. Files served and internal coherence

In [ ]:
# A1. The deployed site serves exactly the repository files
if SITE:
    for name in ["algo1_performance.json", "algo2_performance.json", "algo1.json", "algo2.json"]:
        local = (DASH / "docs/models" / name).read_bytes()
        try:
            online = urllib.request.urlopen(urllib.request.Request(SITE + "models/" + name, headers={"Cache-Control": "no-cache"}), timeout=30).read()
            record("A", "-", f"online {name} == repository file", hashlib.sha256(online).hexdigest()[:12], hashlib.sha256(local).hexdigest()[:12], exact=True)
        except Exception as e:
            print("could not fetch", name, e)

In [ ]:
# A2. Internal coherence of each performance file (counts -> rates -> AUC -> DCA -> histogram)
def internal(a, d, tag=""):
    n, e = d["n"], d["n_events"]
    r = d["roc"]
    record("A", a+tag, "prevalence = events / n", d["prevalence"], e / n, 1e-12)
    record("A", a+tag, "AUC = area under the stored ROC (trapezoid)", d["auc"], _trapz(r["tpr"], r["fpr"]), 2e-4)
    bad = 0
    for g in d["grid"]:
        tp, fp, fn, tn = g["tp"], g["fp"], g["fn"], g["tn"]
        ok = tp + fn == e and fp + tn == n - e and abs(g["flagged"] - (tp + fp) / n) < 1e-9
        ok &= abs(g["sens"] - tp / e) < 1e-9 and abs(g["spec"] - tn / (n - e)) < 1e-9
        if tp + fp: ok &= abs(g["ppv"] - tp / (tp + fp)) < 1e-9
        if tn + fn: ok &= abs(g["npv"] - tn / (tn + fn)) < 1e-9
        bad += not ok
    record("A", a+tag, "199 threshold rows: TP+FN=events, FP+TN=non-events, sens/spec/PPV/NPV = counts ratios", bad, 0, exact=True)
    grid = {round(g["t"], 3): g for g in d["grid"]}
    bad = 0
    for x in d["dca"]:
        g = grid[round(x["t"], 3)] if round(x["t"], 3) in grid else min(d["grid"], key=lambda g: abs(g["t"] - x["t"]))
        w = x["t"] / (1 - x["t"])
        bad += abs(x["model"] - (g["tp"] / n - g["fp"] / n * w)) > 1e-9 or abs(x["all"] - (e / n - (n - e) / n * w)) > 1e-9
    record("A", a+tag, "DCA net benefit = TP/n - FP/n * t/(1-t) from the threshold table", bad, 0, exact=True)
    record("A", a+tag, "histogram counts sum to non-events / events", [sum(d["hist"]["neg"]), sum(d["hist"]["pos"])], [n - e, e], exact=False, tol=0)
    record("A", a+tag, "calibration decile sizes sum to n", sum(b["n"] for b in d["calibration"]["bins"]), n, exact=True)

for a, d in PERF.items():
    internal(a, d)
    c = d.get("comparators", {}).get("tabpfn")
    if c: internal(a, c, " [TabPFN]")
pd.DataFrame(results, columns=["level", "algo", "check", "shown", "recomputed", "ok"]).query("level == 'A'")[["algo", "check", "ok"]]

## B. Recompute from the individual-level predictions

Run these two lines at the **end of the export notebooks** (they use objects already in memory), then come back here.
This writes the *same* predictions that fed the dashboard JSON, to the private folder:

```python
# export_modele.ipynb (algo 1), after the cell that defines y_te_ and p_te
pd.DataFrame({"y": y_te_, "p": p_te}).to_csv(PRIVATE / "algo1_pred.csv", index=False)

# 2-algo2-export-train.ipynb (algo 2), after the cell that defines y and p_cal_oof
import pandas as pd
pd.DataFrame({"y": np.asarray(y).astype(int), "p": p_cal_oof}).to_csv(PRIVATE / "algo2_pred.csv", index=False)
```

In [ ]:
PRED = {}
for a in ALGOS:
    f = PRIVATE / f"{a}_pred.csv"
    if f.exists():
        df = pd.read_csv(f); PRED[a] = (df["y"].to_numpy().astype(int), df["p"].to_numpy(float))
        print(a, "loaded", len(df), "rows,", int(df["y"].sum()), "events")
    else:
        print(a, "MISSING:", f)

In [ ]:
GRID = np.round(np.arange(0.005, 1.0, 0.005), 3)

def fast_delong_ci(y, p):
    """DeLong (1988) analytic 95 % CI of the AUC: an independent method from the bootstrap on the site."""
    pos, neg = p[y == 1], p[y == 0]
    m, n = len(pos), len(neg)
    v10 = np.array([(np.sum(x > neg) + 0.5 * np.sum(x == neg)) / n for x in pos])
    v01 = np.array([(np.sum(pos > x) + 0.5 * np.sum(pos == x)) / m for x in neg])
    auc = v10.mean(); se = np.sqrt(np.var(v10, ddof=1) / m + np.var(v01, ddof=1) / n)
    return auc - 1.96 * se, auc + 1.96 * se

def wilson(k, n, z=1.96):
    ph = k / n; d = 1 + z**2 / n; c = (ph + z**2 / (2 * n)) / d
    h = z * np.sqrt(ph * (1 - ph) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h

def verify(a, y, p, d, tag="", boot_seed=12345):
    n, e = len(y), int(y.sum())
    record("B", a+tag, "n", d["n"], n, exact=True); record("B", a+tag, "events", d["n_events"], e, exact=True)
    # --- AUC and CI
    auc = roc_auc_score(y, p)
    record("B", a+tag, "AUC (sklearn)", d["auc"], auc, 1e-9)
    lo, hi = fast_delong_ci(y, p)
    record("B", a+tag, "AUC 95% CI vs DeLong (tolerance 0.01: bootstrap vs analytic)", d["auc_ci"], [lo, hi], 0.01)
    rng = np.random.default_rng(boot_seed); bs = []
    for _ in range(2000):
        i = rng.integers(0, n, n)
        if y[i].min() != y[i].max(): bs.append(roc_auc_score(y[i], p[i]))
    record("B", a+tag, "AUC 95% CI vs my own bootstrap (other seed, tol 0.01)", d["auc_ci"], np.percentile(bs, [2.5, 97.5]), 0.01)
    # --- ROC curve
    f, t, _ = roc_curve(y, p)
    fs, ts = np.array(d["roc"]["fpr"]), np.array(d["roc"]["tpr"])
    mine_pts = set(zip(np.round(f, 5), np.round(t, 5)))
    record("B", a+tag, "every stored ROC point is a vertex of my ROC curve", sum((x, y_) not in mine_pts for x, y_ in zip(fs.round(5), ts.round(5))), 0, exact=True)
    # --- threshold table (rule: flagged if p >= t)
    bad_counts = 0; maxdev = 0
    for g in d["grid"]:
        fl = p >= g["t"]
        tp = int((fl & (y == 1)).sum()); fp = int((fl & (y == 0)).sum())
        if (tp, fp, e - tp, (n - e) - fp) != (g["tp"], g["fp"], g["fn"], g["tn"]): bad_counts += 1
        for k, v in (("sens", tp / e), ("spec", ((n - e) - fp) / (n - e)), ("ppv", tp / (tp + fp) if tp + fp else None), ("npv", ((n - e) - fp) / (n - tp - fp) if n - tp - fp else None)):
            if v is not None and g[k] is not None: maxdev = max(maxdev, abs(v - g[k]))
    record("B", a+tag, "199 thresholds: TP/FP/FN/TN identical to my counts", bad_counts, 0, exact=True)
    record("B", a+tag, "sens/spec/PPV/NPV max deviation", maxdev, 0, 1e-9)
    # --- calibration
    cal = d["calibration"]
    lp = np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    lr = LogisticRegression(C=1e8, max_iter=2000).fit(lp.reshape(-1, 1), y)
    record("B", a+tag, "calibration slope", cal["slope"], lr.coef_[0, 0], 1e-4)
    record("B", a+tag, "calibration intercept", cal["intercept"], lr.intercept_[0], 1e-4)
    record("B", a+tag, "Brier score", cal["brier"], brier_score_loss(y, p), 1e-9)
    record("B", a+tag, "log loss", cal["log_loss"], log_loss(y, p), 1e-9)
    record("B", a+tag, "mean predicted probability", cal["mean_p"], p.mean(), 1e-9)
    o = np.argsort(p, kind="stable"); parts = np.array_split(o, 10)   # independent decile definition (equal sizes)
    mine = [(p[i].mean(), y[i].mean(), len(i)) for i in parts]
    record("B", a+tag, "decile sizes (shown vs equal-size deciles, tolerance 1 person)", [b["n"] for b in cal["bins"]], [m[2] for m in mine], 1)
    record("B", a+tag, "decile mean predicted prob (tol 0.01)", [b["p"] for b in cal["bins"]], [m[0] for m in mine], 0.01)
    record("B", a+tag, "decile observed rate (tol 0.03)", [b["obs"] for b in cal["bins"]], [m[1] for m in mine], 0.03)
    # --- decision curve
    bad = 0
    for x in d["dca"]:
        fl = p >= x["t"]; w = x["t"] / (1 - x["t"])
        nb = (fl & (y == 1)).sum() / n - (fl & (y == 0)).sum() / n * w
        bad += abs(nb - x["model"]) > 1e-9
    record("B", a+tag, "DCA net benefit (60 thresholds)", bad, 0, exact=True)
    # --- histograms
    edges = np.array(d["hist"]["edges"])
    record("B", a+tag, "histogram, non-events", d["hist"]["neg"], np.histogram(p[y == 0], np.linspace(0, 1, 41))[0], 0)
    record("B", a+tag, "histogram, events", d["hist"]["pos"], np.histogram(p[y == 1], np.linspace(0, 1, 41))[0], 0)
    return auc

for a, (y, p) in PRED.items():
    verify(a, y, p, PERF[a])
    c = PERF[a].get("comparators", {}).get("tabpfn")
    tf = PRIVATE / f"{a}_tabpfn.csv"
    if c and tf.exists():
        t = pd.read_csv(tf); assert (t["_y"].to_numpy() == y).all(), "TabPFN rows are not aligned with the main model"
        verify(a, y, t["p_tabpfn"].to_numpy(), c, " [TabPFN]")
        # paired AUC difference
        dd = roc_auc_score(y, t["p_tabpfn"]) - roc_auc_score(y, p)
        record("B", a, "TabPFN - main AUC difference", c["auc_diff_vs_main"]["diff"], dd, 1e-9)
df = pd.DataFrame(results, columns=["level", "algo", "check", "shown", "recomputed", "ok"])
df.query("level == 'B'")

### What the Predict and Performance tabs display
The site reads each number from the JSON with these rules. This cell applies them **independently** to your recomputed predictions, so that what you see on screen (default threshold, risk zones and their composition) can be compared line by line.

In [ ]:
def site_view(a, d, y=None, p=None):
    t = d["default_threshold"]
    g = min(d["grid"], key=lambda g: abs(g["t"] - t))      # the site rounds the threshold to the nearest 0.005 grid point
    print(f"--- {a}: AUC {d['auc']:.3f} [{d['auc_ci'][0]:.3f}-{d['auc_ci'][1]:.3f}]  n={d['n']}  prevalence {d['prevalence']:.1%}")
    print(f"default threshold {t:.3f}; the site reads the grid at {g['t']:.3f}:  sens {g['sens']:.3f}  spec {g['spec']:.3f}  PPV {g['ppv']:.3f}  NPV {g['npv']:.3f}")
    if y is not None:
        fl = p >= t; tp = (fl & (y == 1)).sum(); fp = (fl & (y == 0)).sum(); e = y.sum(); n = len(y)
        print(f"   exact at {t:.3f}                 :  sens {tp/e:.3f}  spec {1-fp/(n-e):.3f}  PPV {tp/(tp+fp):.3f}  NPV {(n-e-fp)/(n-tp-fp):.3f}   <- difference = grid rounding only")
    lo = max(x["t"] for x in d["grid"] if x["sens"] is not None and x["sens"] >= 0.90)
    hi = min(x["t"] for x in d["grid"] if x["spec"] is not None and x["spec"] >= 0.90)
    gl, gh = (min(d["grid"], key=lambda g: abs(g["t"] - v)) for v in (lo, hi))
    n, E = d["n"], d["n_events"]
    zones = {"low": (gl["fn"] + gl["tn"], gl["fn"]), "high": (gh["tp"] + gh["fp"], gh["tp"])}
    zones["mid"] = (n - zones["low"][0] - zones["high"][0], E - zones["low"][1] - zones["high"][1])
    print(f"risk zones: rule-out < {lo:.3f} (largest t with sens >= 90 %), rule-in >= {hi:.3f} (smallest t with spec >= 90 %)")
    for k in ("low", "mid", "high"):
        m, ev = zones[k]; l, h = wilson(ev, m)
        print(f"   {k:<4} n={m:4d} ({m/n:5.1%})  PTSD {ev:3d} -> observed {ev/m:5.1%}  (Wilson 95 % CI {l:.1%}-{h:.1%})")
for a, d in PERF.items():
    site_view(a, d, *PRED.get(a, (None, None)))

## C. The browser engine against Python

The site's JavaScript re-implements the XGBoost trees (with exact TreeSHAP) and the logistic model (with Platt calibration).
`tools/check_js.mjs` compares it with `xgboost.pred_contribs` and with `predict.py` on the reference rows in `tools/reference/`.

In [ ]:
node = shutil.which("node")
if node:
    r = subprocess.run([node, "tools/check_js.mjs"], cwd=DASH, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-500:])
    record("C", "-", "check_js.mjs exits OK (probabilities and SHAP identical to Python)", r.returncode, 0, exact=True)
else:
    print("node not found: install Node.js or run `node tools/check_js.mjs` in a terminal")

## Summary

In [ ]:
df = pd.DataFrame(results, columns=["level", "algo", "check", "shown", "recomputed", "ok"])
print(df.groupby("level")["ok"].agg(checks="size", passed="sum"))
failed = df[~df["ok"]]
print("\nALL CHECKS PASSED" if failed.empty and (df.level == "B").any() else "\nFAILED / MISSING:")
failed[["level", "algo", "check", "shown", "recomputed"]] if not failed.empty else (None if (df.level == "B").any() else print("Level B not run: algoX_pred.csv missing"))